## Your pipeline, part 4: a stronger third model (YOLO11m) + 3-model fusion
HW2, notebook 10 (Kaggle GPU, ~3.5-4 h). Goal: leaderboard ~0.30 (now 0.2765 with notebook 09's 2-model fusion).

**Why.** Notebook 09 fused two YOLO11s models (+0.020 val mAP over the better one, +0.007 on the leaderboard). Both are the
same small network, so their errors are alike. A model that is both **stronger and different** adds the most to a fusion:
**YOLO11m** (~3x the compute of YOLO11s; on VisDrone a larger YOLO adds about +0.02-0.03 mAP as a single model,
mainly on small and dense objects).

**Steps**
1. `L1_m960_e12_cos` on fold 0: YOLO11m with 04's best config (960 px, box 12, dfl 2), **12 epochs** with a cosine schedule
   (08's held-out curve peaked at epochs 12-13 of 20). Compared with YOLO11s on the same fold (04 and 08 rows). `RUN_CV = False` skips it (~1 h less).
2. Final YOLO11m on all training frames -> `final_10.pt` (~1.5 h).
3. Its six passes (`A2+A3`) on val and test, saved as `passes_10_*.pkl.gz` (~30-45 min).
4. Fusion on val with the method of notebook 09, now across 3 models (`final_04`, `final_08`, `final_10`; plus `final_05`
   RT-DETR if its output is attached: a different architecture, the fusion search decides whether it helps).
   Check first: the 2-model fusion of 09 must be reproduced exactly.
5. Test: fuse the members' test outputs -> `submission_10_fusion.csv` (+ `submission_10_yolo11m.csv`, YOLO11m alone).

**Search on val, small and staged** (7 val clips): score power `p` in {1, 2, 3} at equal weights -> one member up-weighted (x2)
or left out, each in turn -> matching IoU in {0.5, 0.6}.

**Go / no-go:** submit only if the best fusion beats notebook 09's fusion on val by at least **+0.005**
(09: +0.020 on val gave +0.007 on the leaderboard).

**Inputs:** competition; outputs of **01** (`vd_prep`), **04** (`final_04.pt`, `results_04_folds.csv`), **07** (`submission_07_A2_A3_wbf.csv`,
`results_07_inference.csv`), **08** (`final_08.pt`, `submission_08_A2_A3_wbf.csv`, `passes_08_val.pkl.gz`, `results_08_*.csv`),
**09** (`results_09_val.csv`, `passes_final_04_val.pkl.gz`). Optional: **05** (`final_05.pt`). GPU T4, **Internet ON**.

Protections as in 04 / 08: training in separate processes, chunked prediction, a GPU memory check first, finished runs,
final weights and passes are reused from an attached earlier version of this notebook, results are saved after every step.

In [1]:
%pip install -q ultralytics

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less GPU memory fragmentation

import os, io, json, time, math, random, shutil, contextlib
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

SEED = 0
def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> turn on a GPU in the session options")

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")

def find_all(pattern, depth=5):
    # search attached inputs at increasing nesting depth (Kaggle nests inputs differently per source)
    for d in range(depth):
        hit = sorted(INPUT.glob("/".join(["*"] * d + [pattern])))
        if hit:
            return hit
    return []

def find(pattern):
    hit = find_all(pattern)
    if not hit:
        raise FileNotFoundError(f"{pattern} not found under {INPUT}: check the attached inputs")
    return hit[0]

PREP = find("vd_prep/config.json").parent            # output of notebook 01
CFG = json.load(open(PREP / "config.json"))
TEST_CSV = find("test_images.csv"); ROOT = TEST_CSV.parent
VAL_DIR = find("val/annotations_coco.json").parent
FOLD = CFG["folds"]                                   # train clip -> fold id (GroupKFold by scene, from notebook 01)
METRICS = ["mAP", "AP50", "AP_s", "AP_m", "AP_l", "ms_per_img"]

import gc, gzip, pickle
from multiprocessing import Pool
from torchvision.ops import batched_nms
from ultralytics import YOLO, RTDETR, settings
from ultralytics.utils import DEFAULT_CFG_DICT
settings.update({"wandb": False})
DEVICE = "cuda"

# ---- training
RUN_CV = True                       # fold-0 run of YOLO11m (comparison with YOLO11s); False saves ~1 h
EPOCHS, BATCH = 12, 4
FOLDS_TO_RUN = [0]
VAL_STEP = 5
NB, TAG, MODEL_NAME, MODEL_CLS = "10", "yolo10m", "YOLO11m pipeline", YOLO
COMMON = dict(epochs=EPOCHS, batch=BATCH, workers=4, device=0, seed=SEED, warmup_epochs=2, close_mosaic=3,
              plots=False, exist_ok=True)
CONFIGS = {"L1_m960_e12_cos": dict(imgsz=960, box=12.0, dfl=2.0, epochs=EPOCHS, cos_lr=True)}   # 04's best config, YOLO11m
make_model = lambda cfg: {"cls": "YOLO", "build": "yolo11m.pt"}     # built inside the worker process
hp_of = lambda cfg: dict(cfg)

# ---- inference and fusion (as in notebooks 07-09)
UP_SCALE, TILE_ZOOM, TILE_OVERLAP = 1.33, 1.5, 0.2
NMS_IOU, WBF_IOU, MAX_DET = 0.6, 0.55, 500
ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]
MERGE_WORKERS = 4                   # CPU processes for WBF / fusion
PREC = {"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True}   # fp16 (renamed in Ultralytics 8.4)
GO_MARGIN = 0.005                   # required val gain over notebook 09's fusion

# ---- inputs: fail here, not after hours
need = {"final_04.pt": "04", "submission_07_A2_A3_wbf.csv": "07", "results_07_inference.csv": "07",
        "submission_08_A2_A3_wbf.csv": "08", "passes_08_val.pkl.gz": "08", "results_08_summary.csv": "08",
        "results_09_val.csv": "09"}
missing = [f"{f} (notebook {src})" for f, src in need.items() if not find_all(f)]
assert not missing, "attach the outputs that contain: " + ", ".join(missing)
if not find_all("passes_final_04_val.pkl.gz"):
    print("passes_final_04_val.pkl.gz (notebook 09) not attached: final_04's val passes will be recomputed (~5 min)")
HAVE_05 = bool(find_all("final_05.pt"))
print("optional RT-DETR member (final_05.pt):", "found" if HAVE_05 else "not attached, fusing 3 models")

GPU: Tesla T4
optional RT-DETR member (final_05.pt): not attached, fusing 3 models


### Data

In [3]:
def load_gt(path):
    # COCO ground truth; ignored regions (iscrowd=1) are copied to all 8 classes so they ignore detections of any class
    d = json.load(open(path))
    crowd = [a for a in d["annotations"] if a.get("iscrowd")]
    d["annotations"] = [a for a in d["annotations"] if not a.get("iscrowd")]
    nid = max([a["id"] for a in d["annotations"]], default=0) + 1
    for a in crowd:
        for c in range(1, 9):
            d["annotations"].append({**a, "id": nid, "category_id": c, "iscrowd": 1, "area": a["bbox"][2] * a["bbox"][3]})
            nid += 1
    gt = COCO()
    gt.dataset = d
    with contextlib.redirect_stdout(io.StringIO()):
        gt.createIndex()
    return gt

def coco_eval(gt, dets, img_ids):
    # standard COCO bbox metrics, up to 500 detections per image (as in the leaderboard)
    if not dets:
        return dict(mAP=0.0, AP50=0.0, AP_s=0.0, AP_m=0.0, AP_l=0.0)
    with contextlib.redirect_stdout(io.StringIO()):
        E = COCOeval(gt, gt.loadRes(dets), "bbox")
        E.params.imgIds = list(img_ids)
        E.params.maxDets = [1, 10, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats                                  # s[1], s[3:6] already use maxDets[-1] = 500
    prec = E.eval["precision"][:, :, :, 0, -1]   # [IoU, recall, class] for all sizes at 500 dets (pycocotools' s[0] hardcodes 100)
    prec = prec[prec > -1]
    return dict(mAP=float(prec.mean()) if prec.size else -1.0, AP50=s[1], AP_s=s[3], AP_m=s[4], AP_l=s[5])

def write_submission(dets, path):
    # one row per test image: "class score x y w h ..." (0-based class, pixels, top-left corner, max 500 boxes)
    test = pd.read_csv(TEST_CSV)
    by = {}
    for d in dets:
        by.setdefault(d["image_id"], []).append(d)
    rows = []
    for iid in test.image_id:
        top = sorted(by.get(iid, []), key=lambda d: -d["score"])[:500]
        rows.append(" ".join(f"{d['category_id'] - 1} {d['score']:.4f} " + " ".join(f"{v:.1f}" for v in d["bbox"]) for d in top))
    sub = pd.DataFrame({"image_id": test.image_id, "PredictionString": rows})
    sub.to_csv(path, index=False)
    sample = find_all("sample_submission.csv")      # not shipped with this competition's data: ids come from test_images.csv
    if sample:
        assert list(sub.image_id) == list(pd.read_csv(sample[0]).image_id), "image ids differ from sample_submission.csv"
    print(f"saved {path}: {len(sub)} rows, {sum(map(bool, rows))} with detections, {sum(len(r.split()) // 6 for r in rows)} boxes")

In [4]:
# training frames exported by notebook 01 (ignored regions already grayed out)
TJ = json.load(open(PREP / "train_coco.json"))
IMGS = {i["id"]: i for i in TJ["images"]}
GT_TRAIN = load_gt(PREP / "train_coco.json")

def fold_ids(f):
    # (train ids, held-out ids) for fold f; f=None -> all frames for training
    return ([i for i, m in IMGS.items() if FOLD[m["video"]] != f],
            [i for i, m in IMGS.items() if FOLD[m["video"]] == f])

def copy_parallel(src, dst, workers=16):
    # copy a folder file by file with many threads (on network storage the per-file latency dominates)
    from concurrent.futures import ThreadPoolExecutor
    files = [p for p in Path(src).rglob("*") if p.is_file()]
    def cp(p):
        t = Path(dst) / p.relative_to(src)
        t.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(p, t)
    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(cp, files))

# /kaggle/input is network storage (~5 MB/s, ~50 ms per file): copy the training images once to local disk
IMG_ROOT = PREP / "images" / "train"
if str(PREP).startswith("/kaggle/input"):
    local = Path("/tmp/vd_prep/images/train")
    if not (local.parent / ".copied").exists():
        t0 = time.time()
        copy_parallel(IMG_ROOT, local)
        (local.parent / ".copied").touch()
        print(f"copied {sum(1 for _ in local.iterdir())} training images to local disk in {time.time() - t0:.0f} s")
    IMG_ROOT = local

def train_path(i):
    return IMG_ROOT / IMGS[i]["file_name"]

# official val: raw frames at original resolution, every VAL_STEP-th frame of each clip (for speed)
VAL_GT = load_gt(VAL_DIR / "annotations_coco.json")
_idx = {(p.parent.name, p.name): p for p in (VAL_DIR / "images").rglob("*.jpg")}
_v = pd.DataFrame(VAL_GT.dataset["images"])
_v = _v[(_v.groupby("video").frame_id.rank(method="first").astype(int) - 1) % VAL_STEP == 0]
VAL_ITEMS = [(int(r.id), _idx[(r.video, Path(r.file_name).name)]) for r in _v.itertuples()]
VAL_IDS = [i for i, _ in VAL_ITEMS]
TEST_ITEMS = [(r.image_id, ROOT / r.file_name) for r in pd.read_csv(TEST_CSV).itertuples()]
print(f"{len(IMGS)} train frames | {len(VAL_ITEMS)} val frames | {len(TEST_ITEMS)} test frames")

copied 4868 training images to local disk in 26 s
4868 train frames | 571 val frames | 3811 test frames


In [5]:
import sys, subprocess

DEVICE_ID = globals().get("DEVICE_ID", 0)
# Ultralytics finds labels by swapping images -> labels in the path, and writes a .cache next to them
IMG_DIR = IMG_ROOT                                          # local copy of the training images (data cell)
LBL_DIR = IMG_DIR.parent.parent / "labels" / "train"
if not LBL_DIR.exists():
    copy_parallel(PREP / "labels" / "train", LBL_DIR)
JOB_DIR = Path("/tmp/yolo_jobs") if str(WORK).startswith("/kaggle") else WORK / "yolo_jobs"   # image lists, yamls, job files
JOB_DIR.mkdir(parents=True, exist_ok=True)

def write_data(name, tr_ids, va_ids):
    # Ultralytics data yaml with explicit image lists, so folds are just different lists
    for part, ids in (("train", tr_ids), ("val", va_ids)):
        (JOB_DIR / f"{name}_{part}.txt").write_text("\n".join(str(IMG_DIR / IMGS[i]["file_name"]) for i in ids))
    q = lambda x: json.dumps(str(x))                        # quoted path, safe for Windows backslashes
    p = JOB_DIR / f"{name}.yaml"
    p.write_text(f"path: {q(JOB_DIR)}\ntrain: {q(JOB_DIR / (name + '_train.txt'))}\nval: {q(JOB_DIR / (name + '_val.txt'))}\nnames: {json.dumps(CFG['classes'])}\n")
    return str(p)

# Every training and every prediction runs in its own short-lived Python process. When it exits, all of its
# GPU memory is released, so nothing piles up between runs (inside one process it did, and the GPU ran out).
WORKER = JOB_DIR / "ultra_worker.py"
WORKER.write_text("""
import json, sys
job = json.load(open(sys.argv[1]))
import torch
from ultralytics import YOLO, RTDETR, settings
settings.update({"wandb": False})
Cls = {"YOLO": YOLO, "RTDETR": RTDETR}[job["cls"]]
out = {}
if job["kind"] == "train":
    model = Cls(job["build"])
    if job.get("load"):
        model = model.load(job["load"])                     # built from a yaml: transfer the pretrained weights
    model.train(**job["kwargs"])
else:
    from ultralytics.utils import DEFAULT_CFG_DICT
    kw = dict(job["kwargs"])
    if job.get("fp16"):                                     # newer versions renamed half=True to quantize=16
        kw.update({"quantize": 16} if "quantize" in DEFAULT_CFG_DICT else {"half": True})
    model = Cls(job["weights"])
    items = json.load(open(job["items"]))
    dets, ms, CHUNK = [], [], 16
    # small chunks: Ultralytics 8.4 loads a whole list of paths as ONE batch (ignores `batch`) -> out of memory
    for k in range(0, len(items), CHUNK):
        chunk = items[k:k + CHUNK]
        for (iid, _), r in zip(chunk, model.predict([p for _, p in chunk], stream=True, **kw)):
            b = r.boxes
            xywh = b.xyxy.cpu().numpy()
            xywh[:, 2:] -= xywh[:, :2]
            for box, s, c in zip(xywh.tolist(), b.conf.tolist(), b.cls.tolist()):
                dets.append([iid, int(c) + 1, *[round(v, 2) for v in box], round(s, 5)])
            ms.append(r.speed["inference"] + r.speed["postprocess"])
    out["dets"], out["ms"] = dets, sum(ms) / max(len(ms), 1)
if torch.cuda.is_available():
    out["peak_gb"] = torch.cuda.max_memory_reserved() / 1e9
json.dump(out, open(job["out"], "w"))
""")

def run_worker(job, tag):
    # runs one job in a fresh process, prints its log here, returns its result
    job = {**job, "out": str(JOB_DIR / f"{tag}.out.json")}
    jp = JOB_DIR / f"{tag}.job.json"
    jp.write_text(json.dumps(job))
    proc = subprocess.Popen([sys.executable, str(WORKER), str(jp)], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"{tag}: the worker process failed, see its log above")
    return json.loads(Path(job["out"]).read_text())

def fit(spec, name, tr_ids, va_ids, **hp):
    kw = {"data": write_data(name, tr_ids, va_ids), "project": str(WORK / "runs"), "name": name, **{**COMMON, **hp}}
    res = run_worker({**spec, "kind": "train", "kwargs": kw}, name)
    (WORK / "curves").mkdir(exist_ok=True)
    shutil.copy(WORK / "runs" / name / "results.csv", WORK / "curves" / f"{TAG}_{name}.csv")     # per-epoch val curve
    # last epoch, not "best": best is picked on the held-out fold
    return WORK / "runs" / name / "weights" / "last.pt", res.get("peak_gb", float("nan"))

def detect(weights, items, imgsz):
    # returns COCO-style detections in original pixel coordinates and model ms per image (inference + postprocess)
    tag = f"predict_{Path(weights).parent.parent.name}_{len(items)}"
    ip = JOB_DIR / f"{tag}.items.json"
    ip.write_text(json.dumps([[i, str(p)] for i, p in items]))
    kw = dict(imgsz=imgsz, conf=0.001, iou=0.6, max_det=500, batch=4, verbose=False, device=DEVICE_ID)
    res = run_worker({"cls": MODEL_CLS.__name__, "kind": "predict", "weights": str(weights), "items": str(ip),
                      "kwargs": kw, "fp16": True}, tag)
    dets = [{"image_id": d[0], "category_id": d[1], "bbox": d[2:6], "score": d[6]} for d in res["dets"]]
    return dets, res["ms"]

def eval_fold(weights, ids, imgsz):
    dets, ms = detect(weights, [(i, IMG_DIR / IMGS[i]["file_name"]) for i in ids], imgsz)
    return {**coco_eval(GT_TRAIN, dets, ids), "ms_per_img": ms}

### GPU memory check (~5 min)
One epoch of YOLO11m on the 64 most crowded frames, then a prediction on them at the same size.
If memory is too tight the notebook stops here: lower `BATCH` in the settings cell. Skipped when `final_10.pt` is attached.

In [6]:
if find_all("final_10.pt"):                               # final model attached from an earlier run: nothing left to train
    print("final_10.pt attached: memory check skipped")
else:
    n_obj = {}
    for a in TJ["annotations"]:
        if not a.get("iscrowd"):
            n_obj[a["image_id"]] = n_obj.get(a["image_id"], 0) + 1
    dense = sorted(IMGS, key=lambda i: -n_obj.get(i, 0))[:64]
    heavy = max(CONFIGS.values(), key=lambda c: c["imgsz"])
    kw = {"data": write_data("memcheck", dense, dense[:16]), "project": str(WORK / "runs"), "name": "memcheck",
          **{**COMMON, **hp_of(heavy), "epochs": 1, "warmup_epochs": 0, "close_mosaic": 0}}
    res = run_worker({**make_model(heavy), "kind": "train", "kwargs": kw}, "memcheck")
    mc_dets, _ = detect(WORK / "runs" / "memcheck" / "weights" / "last.pt", [(i, IMG_DIR / IMGS[i]["file_name"]) for i in dense], heavy["imgsz"])
    print(f"prediction check: {len(dense)} frames -> {len(mc_dets)} boxes")
    shutil.rmtree(WORK / "runs" / "memcheck", ignore_errors=True)
    peak, total = res.get("peak_gb", 0.0), torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"peak GPU memory {peak:.1f} of {total:.1f} GB")
    assert peak < 0.85 * total, "too close to the GPU memory limit: lower BATCH in the settings cell and run again"

1 epochs completed in 0.002 hours.
prediction check: 64 frames -> 32000 boxes
peak GPU memory 5.5 of 15.6 GB


### Fold 0: YOLO11m vs YOLO11s (same fold, same evaluator)

In [7]:
if RUN_CV:
    # every config is trained on the same folds (cross-validation over clips) and scored on the held-out clips.
    # Results are saved after every run; runs already in results_{NB}_folds.csv (here, or in an attached
    # earlier version's output) are skipped, so a crash does not cost the finished runs again.
    prev = [p for p in [WORK / f"results_{NB}_folds.csv"] if p.exists()] + find_all(f"results_{NB}_folds.csv")
    rows = pd.read_csv(prev[0]).to_dict("records") if prev else []
    done = {(r["config"], int(r["fold"])) for r in rows}
    if done:
        print("reusing finished runs from", prev[0], ":", sorted(done))
        pd.DataFrame(rows).to_csv(WORK / f"results_{NB}_folds.csv", index=False)     # keep them in this version's output too
    (WORK / "curves").mkdir(exist_ok=True)
    for name, cfg in CONFIGS.items():
        for f in FOLDS_TO_RUN:
            if (name, f) in done:
                for p in find_all(f"curves/{TAG}_{name}_f{f}.csv"):        # keep its training curve for notebook 06
                    if not (WORK / "curves" / p.name).exists():
                        shutil.copy(p, WORK / "curves" / p.name)
                continue
            tr_ids, va_ids = fold_ids(f)
            t0 = time.time()
            run = f"{name}_f{f}"
            trained = [p for p in [WORK / "runs" / run / "weights" / "last.pt"] if p.exists()] + find_all(f"runs/{run}/weights/last.pt")
            curve = [p for p in [WORK / "runs" / run / "results.csv"] if p.exists()] + find_all(f"runs/{run}/results.csv")
            if trained and curve:                                   # trained in an earlier, interrupted version: only evaluate
                print("reusing trained weights", trained[0])
                w, peak = trained[0], float("nan")
                if curve[0] != WORK / "curves" / f"{TAG}_{run}.csv":
                    shutil.copy(curve[0], WORK / "curves" / f"{TAG}_{run}.csv")
            else:
                w, peak = fit(make_model(cfg), run, tr_ids, va_ids, **hp_of(cfg))
            m = eval_fold(w, va_ids, cfg["imgsz"])
            rows.append({"config": name, "fold": f, **m, "train_min": (time.time() - t0) / 60, "peak_gb": peak})
            pd.DataFrame(rows).to_csv(WORK / f"results_{NB}_folds.csv", index=False)
            print(name, "fold", f, {k: round(v, 4) for k, v in m.items()}, f"| peak GPU memory of the training: {peak:.1f} GB")
    folds_df = pd.DataFrame(rows)
    folds_df = folds_df[folds_df.config.isin(CONFIGS) & folds_df.fold.isin(FOLDS_TO_RUN)]
    display(folds_df.groupby("config")[METRICS].agg(["mean", "std"]).round(4))
    cmp = []
    for f, label, cfgs in (("results_04_folds.csv", "04 (YOLO11s, 8 epochs)", None), ("results_08_folds.csv", "08 (YOLO11s, 20 epochs)", None)):
        for p in find_all(f)[:1]:
            d = pd.read_csv(p)
            d = d[d.fold.isin(FOLDS_TO_RUN)]
            cmp += [{"config": f"{label} {c}", **g[METRICS].mean().to_dict()} for c, g in d.groupby("config")]
    cmp += [{"config": c, **folds_df[folds_df.config == c][METRICS].mean().to_dict()} for c in CONFIGS if (folds_df.config == c).any()]
    display(pd.DataFrame(cmp).set_index("config").round(4))
else:
    print("RUN_CV = False: fold-0 run skipped")

12 epochs completed in 0.903 hours.
L1_m960_e12_cos fold 0 {'mAP': 0.2055, 'AP50': np.float64(0.3951), 'AP_s': np.float64(0.0884), 'AP_m': np.float64(0.2635), 'AP_l': np.float64(0.5229), 'ms_per_img': 13.4938} | peak GPU memory of the training: 5.5 GB


mAP        AP50        AP_s        AP_m        AP_l      \
                   mean std    mean std    mean std    mean std    mean std   
config                                                                        
L1_m960_e12_cos  0.2055 NaN  0.3951 NaN  0.0884 NaN  0.2635 NaN  0.5229 NaN   

                ms_per_img      
                      mean std  
config                          
L1_m960_e12_cos    13.4938 NaN

,mAP,AP50,AP_s,AP_m,AP_l,ms_per_img
config,,,,,,
"04 (YOLO11s, 8 epochs) C0_imgsz640",0.1671,0.3138,0.0513,0.2078,0.4594,4.1696
"04 (YOLO11s, 8 epochs) C1_imgsz960",0.2030,0.3836,0.0800,0.2658,0.4741,6.4068
"04 (YOLO11s, 8 epochs) C2_imgsz960_lossgains",0.2018,0.3883,0.0835,0.2678,0.4662,6.8313
"08 (YOLO11s, 20 epochs) M1_e20_cos",0.1881,0.3623,0.0742,0.2396,0.5071,6.4516
"08 (YOLO11s, 20 epochs) M2_1280_e20_cos",0.1942,0.3826,0.0868,0.2596,0.4481,10.7229
L1_m960_e12_cos,0.2055,0.3951,0.0884,0.2635,0.5229,13.4938


**Observations:**
- On fold 0 YOLO11m (12 epochs, cosine) is only slightly better than the best YOLO11s: mAP 0.206 vs 0.203 (04, 960 px), AP_small 0.088 vs 0.080-0.087, AP_large 0.523 vs 0.474.
- It costs about 2x per image (13.5 vs 6.4-6.8 ms) and 3x training time (0.90 h vs 0.31 h).

### Final YOLO11m on all training frames

In [8]:
FINAL_CFG = CONFIGS["L1_m960_e12_cos"]
done_final = [p for p in [WORK / "final_10.pt"] if p.exists()] + find_all("final_10.pt")
if done_final:                                              # trained in an earlier run of this notebook
    W_FINAL = done_final[0]
    print("reusing", W_FINAL)
else:
    run = "final_10"
    trained = [p for p in [WORK / "runs" / run / "weights" / "last.pt"] if p.exists()] + find_all(f"runs/{run}/weights/last.pt")
    if trained and ([p for p in [WORK / "runs" / run / "results.csv"] if p.exists()] + find_all(f"runs/{run}/results.csv")):
        w = trained[0]
        print("reusing trained weights", w)
    else:
        tr_all, _ = fold_ids(None)
        w, peak = fit(make_model(FINAL_CFG), run, tr_all, fold_ids(0)[1][:200], **hp_of(FINAL_CFG))  # 200 held-out frames only feed the log
        print(f"final model trained | peak GPU memory {peak:.1f} GB")
    W_FINAL = WORK / "final_10.pt"
    shutil.copy(w, W_FINAL)
if W_FINAL != WORK / "final_10.pt":
    shutil.copy(W_FINAL, WORK / "final_10.pt")              # keep it in this version's output too

12 epochs completed in 1.085 hours.
final model trained | peak GPU memory 5.4 GB


### Prediction passes and merging (code of notebooks 07-09)

In [9]:
def load_model(path):
    # model + the image size it was trained at
    ck = torch.load(path, map_location="cpu", weights_only=False)
    imgsz = int(np.max(ck.get("train_args", {}).get("imgsz", 960)))
    net = ck.get("model") or ck.get("ema")                  # last.pt keeps the weights under "ema"
    cls = RTDETR if type(net).__name__.startswith("RTDETR") else YOLO
    del ck, net
    return cls(str(path)), imgsz

def run(model, images, imgsz):
    # one model call on a few images (Ultralytics 8.4 puts a whole list into one batch); one (N, 6) array per image
    res = model.predict(images, imgsz=imgsz, conf=0.001, iou=0.6, max_det=MAX_DET, verbose=False, device=DEVICE_ID, **PREC)
    return [np.c_[r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy()].astype(np.float32) for r in res]

def unflip(b, width):
    # boxes predicted on a horizontally flipped image -> original orientation
    b = b.copy(); b[:, [0, 2]] = width - b[:, [2, 0]]
    return b

def nms(b, iou):
    # class-wise NMS, best first, at most MAX_DET boxes (main process only)
    if not len(b):
        return b
    t = torch.from_numpy(b)
    keep = batched_nms(t[:, :4], t[:, 4], t[:, 5].long(), iou).numpy()
    return b[keep[:MAX_DET]]

def tile_grid(W, H):
    # overlapping crops of side max(W, H) / TILE_ZOOM (clipped to the frame), at least TILE_OVERLAP overlap
    T = int(max(W, H) / TILE_ZOOM)
    tw, th = min(T, W), min(T, H)
    def starts(size, t):
        n = max(1, math.ceil((size - t) / (t * (1 - TILE_OVERLAP))) + 1) if size > t else 1
        return np.linspace(0, size - t, n).round().astype(int)
    return [(x, y, x + tw, y + th) for y in starts(H, th) for x in starts(W, tw)]

ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]

def predict_views(model, imgsz, img):
    # all six passes of one frame, boxes in original pixel coordinates
    H, W = img.shape[:2]
    up = int(round(imgsz * UP_SCALE / 32) * 32)
    flip = np.ascontiguousarray(img[:, ::-1])
    out = {"full": run(model, [img], imgsz)[0], "full_flip": unflip(run(model, [flip], imgsz)[0], W),
           "up": run(model, [img], up)[0], "up_flip": unflip(run(model, [flip], up)[0], W)}
    grid = tile_grid(W, H)
    for name in ("tiles", "tiles_flip"):
        crops = [img[y0:y1, x0:x1] for x0, y0, x1, y1 in grid]
        if name == "tiles_flip":
            crops = [np.ascontiguousarray(c[:, ::-1]) for c in crops]
        parts = []
        for (x0, y0, x1, y1), b in zip(grid, run(model, crops, imgsz)):
            if name == "tiles_flip":
                b = unflip(b, x1 - x0)
            # drop boxes cut by an inner crop edge: the neighbouring crop (>= 20 % overlap) or the full pass has the whole object
            e = 2
            cut = (((b[:, 0] <= e) & (x0 > 0)) | ((b[:, 2] >= x1 - x0 - e) & (x1 < W)) |
                   ((b[:, 1] <= e) & (y0 > 0)) | ((b[:, 3] >= y1 - y0 - e) & (y1 < H)))
            b = b[~cut].copy(); b[:, [0, 2]] += x0; b[:, [1, 3]] += y0
            parts.append(b)
        out[name] = nms(np.concatenate(parts), NMS_IOU)       # remove duplicates from overlapping crops
    return out

def passes(model_path, tag, items):
    # prediction passes of one model on a set of frames; reused from this or an attached output, else computed and saved
    fname = f"passes_{tag}.pkl.gz"
    hit = [p for p in [WORK / fname] if p.exists()] + find_all(fname)
    if hit:
        with gzip.open(hit[0], "rb") as f:
            cache = pickle.load(f)
        if set(cache) == {i for i, _ in items}:
            print(f"reusing {hit[0]}")
            return cache
        print(f"{hit[0]} covers other frames, recomputing")
    model, imgsz = load_model(model_path)
    print(f"{tag}: {Path(model_path).name} trained at {imgsz} px, {len(items)} frames")
    cache = {iid: predict_views(model, imgsz, cv2.imread(str(p))) for iid, p in tqdm(items, desc=tag)}
    del model; gc.collect(); torch.cuda.empty_cache()
    with gzip.open(WORK / fname, "wb") as f:
        pickle.dump(cache, f, protocol=4)
    return cache

In [10]:
def wbf(views, iou_thr=WBF_IOU):
    # views: list of (N, 6) arrays, one per pass; returns fused (M, 6), best first, at most MAX_DET
    n_views = len(views)
    allb = np.concatenate([v for v in views if len(v)] or [np.zeros((0, 6), np.float32)])
    out = []
    for c in np.unique(allb[:, 5]):
        b = allb[allb[:, 5] == c]
        b = b[np.argsort(-b[:, 4])]
        acc = np.zeros((len(b), 4)); fb = np.zeros((len(b), 4)); w = np.zeros(len(b)); cnt = np.zeros(len(b)); m = 0
        for x1, y1, x2, y2, s, _ in b:
            if m:
                f = fb[:m]
                ix = np.clip(np.minimum(f[:, 2], x2) - np.maximum(f[:, 0], x1), 0, None)
                iy = np.clip(np.minimum(f[:, 3], y2) - np.maximum(f[:, 1], y1), 0, None)
                inter = ix * iy
                iou = inter / ((f[:, 2] - f[:, 0]) * (f[:, 3] - f[:, 1]) + (x2 - x1) * (y2 - y1) - inter + 1e-9)
                j = int(np.argmax(iou))
                if iou[j] > iou_thr:
                    acc[j] += s * np.array([x1, y1, x2, y2]); w[j] += s; cnt[j] += 1
                    fb[j] = acc[j] / w[j]
                    continue
            acc[m] = s * np.array([x1, y1, x2, y2]); w[m] = s; cnt[m] = 1; fb[m] = (x1, y1, x2, y2); m += 1
        score = w[:m] / cnt[:m] * np.minimum(cnt[:m], n_views) / n_views
        out.append(np.c_[fb[:m], score, np.full(m, c)])
    if not out:
        return np.zeros((0, 6), np.float32)
    res = np.concatenate(out).astype(np.float32)
    return res[np.argsort(-res[:, 4])][:MAX_DET]

def merge(views, how):
    if how is None:                                         # single pass: as predicted
        return views[0][:MAX_DET]
    if how == "nms":
        return nms(np.concatenate(views), NMS_IOU)
    return wbf(views)

STRATEGIES = {                                              # name: (views, merge)
    "S0_single_pass":   (["full"], None),
    "A1_upscaled_pass": (["up"], None),
    "A2_tta_nms":       (["full", "full_flip", "up", "up_flip"], "nms"),
    "A2_tta_wbf":       (["full", "full_flip", "up", "up_flip"], "wbf"),
    "A3_tiles_nms":     (["full", "tiles"], "nms"),
    "A3_tiles_wbf":     (["full", "tiles"], "wbf"),
    "A2+A3_wbf":        (["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"], "wbf"),
}
ALL_VIEWS = ["full", "full_flip", "up", "up_flip", "tiles", "tiles_flip"]

def merge_task(args):
    # one image, one strategy (runs in a worker process)
    iid, views, how = args
    t0 = time.perf_counter()
    b = merge(views, how)
    return iid, b, 1000 * (time.perf_counter() - t0)

def merge_all(cache, views, how):
    # merge the cached passes of every frame; returns detections (COCO style) and merge ms per frame
    tasks = [(iid, [cache[iid][v] for v in views], how) for iid in cache]
    # worker processes only for WBF (pure NumPy). NMS uses PyTorch, which can hang inside forked processes
    # once its thread pool was used in the main process; NMS takes only seconds here anyway.
    if MERGE_WORKERS and how == "wbf":
        with Pool(MERGE_WORKERS) as p:
            res = p.map(merge_task, tasks, chunksize=8)
    else:
        res = [merge_task(t) for t in tasks]
    dets = [{"image_id": iid, "category_id": int(r[5]) + 1, "bbox": [float(r[0]), float(r[1]), float(r[2] - r[0]), float(r[3] - r[1])],
             "score": float(r[4])} for iid, b, _ in res for r in b]
    return dets, float(np.mean([t for _, _, t in res]))

In [11]:
def pmap(f, tasks, chunk):
    # NumPy-only work in MERGE_WORKERS processes (serial with 0)
    if not MERGE_WORKERS:
        return [f(t) for t in tasks]
    with Pool(MERGE_WORKERS) as p:
        return p.map(f, tasks, chunksize=chunk)

def merged(cache, ids):
    # six-pass WBF of each image (strategy A2+A3_wbf), rounded like the submission CSVs (score 4, box 1 decimal)
    tasks = [(iid, [cache[iid][v] for v in ALL_VIEWS], "wbf") for iid in ids]
    res = pmap(merge_task, tasks, 8)
    return {iid: csv_round(b) for iid, b, _ in res}

def csv_round(b):
    # same precision as a submission row: the val numbers then match what the test CSVs contain
    b = b.copy().astype(np.float64)
    wh = np.round(b[:, 2:4] - b[:, 0:2], 1); b[:, 0:2] = np.round(b[:, 0:2], 1); b[:, 2:4] = b[:, 0:2] + wh
    b[:, 4] = np.round(b[:, 4], 4)
    return b.astype(np.float32)

def to_dets(by):
    return [{"image_id": iid, "category_id": int(r[5]) + 1, "bbox": [float(r[0]), float(r[1]), float(r[2] - r[0]), float(r[3] - r[1])],
             "score": float(r[4])} for iid, b in by.items() for r in b]


def read_submission(path):
    # submission CSV -> {image_id: (N, 6) [x1, y1, x2, y2, score, class]}
    out = {}
    for iid, s in pd.read_csv(path).fillna("").itertuples(index=False):
        v = np.array(s.split(), dtype=np.float64).reshape(-1, 6)       # class score x y w h
        out[iid] = np.c_[v[:, 2], v[:, 3], v[:, 2] + v[:, 4], v[:, 3] + v[:, 5], v[:, 1], v[:, 0]].astype(np.float32)
    return out

### Members of the fusion: val and test outputs
Every member is the merged (`A2+A3_wbf`, at most 500 boxes) output of one model. 04 and 08 on test: their submission CSVs.

In [12]:
def member_passes(weights, tag, items):
    # saved passes if attached (or made earlier in this run), otherwise computed with the weights
    w = (find_all(weights) or [Path(weights)])[0]
    return passes(w, tag, items)

TEST_ITEMS = [(r.image_id, ROOT / r.file_name) for r in pd.read_csv(TEST_CSV).itertuples()]
VAL_IDS_SET, TEST_IDS = set(VAL_IDS), [i for i, _ in TEST_ITEMS]
VAL, TEST = {}, {}
for name, weights, tag in (("04", "final_04.pt", "final_04_val"), ("08", "final_08.pt", "08_val")):
    c = member_passes(weights, tag, VAL_ITEMS); VAL[name] = merged(c, VAL_IDS); del c
TEST["04"] = read_submission(find("submission_07_A2_A3_wbf.csv"))
TEST["08"] = read_submission(find("submission_08_A2_A3_wbf.csv"))

t0 = time.time()
c = passes(W_FINAL, "10_val", VAL_ITEMS); VAL["10"] = merged(c, VAL_IDS); del c
c = passes(W_FINAL, "10_test", TEST_ITEMS); TEST["10"] = merged(c, TEST_IDS); del c
print(f"YOLO11m passes on val + test: {(time.time() - t0) / 60:.1f} min")
write_submission(to_dets(TEST["10"]), WORK / "submission_10_yolo11m.csv")

if HAVE_05:                                                 # optional: RT-DETR from notebook 05
    c = member_passes("final_05.pt", "final_05_val", VAL_ITEMS); VAL["05"] = merged(c, VAL_IDS); del c
    c = member_passes("final_05.pt", "final_05_test", TEST_ITEMS); TEST["05"] = merged(c, TEST_IDS); del c
gc.collect(); torch.cuda.empty_cache()
for k in TEST:
    assert set(TEST[k]) == set(TEST_IDS), f"member {k}: test output covers other images than test_images.csv"
print("members:", list(VAL))

reusing /kaggle/input/notebooks/vivtrea/09-model-fusion/passes_final_04_val.pkl.gz
reusing /kaggle/input/notebooks/vivtrea/pipeline-training/passes_08_val.pkl.gz
10_val: final_10.pt trained at 960 px, 571 frames
10_test: final_10.pt trained at 960 px, 3811 frames
saved /kaggle/working/submission_10_yolo11m.csv: 3811 rows, 3811 with detections, 1540945 boxes
members: ['04', '08', '10']


### Fusion across the members (notebook 09's rule, for any number of models)
A fused box takes at most one box per model; coordinates are the score x weight average; the score is the weighted mean
score of the models that found it, times (weight share of those models) ^ `p`.

In [13]:
def fuse(parts, w, iou_thr, p):
    # parts: one (N, 6) array [x1, y1, x2, y2, score, class] per model; w: model weights (members with weight 0 are left out)
    keep = [k for k in range(len(parts)) if w[k] > 0]
    parts, w = [parts[k] for k in keep], np.asarray([w[k] for k in keep], float)
    w = w / w.sum(); M = len(parts)
    allb = np.concatenate([np.c_[a, np.full(len(a), k)] for k, a in enumerate(parts)]) if sum(map(len, parts)) else np.zeros((0, 7))
    out = []
    for c in np.unique(allb[:, 5]):
        x = allb[allb[:, 5] == c]
        x = x[np.argsort(-x[:, 4] * w[x[:, 6].astype(int)], kind="stable")]   # best weighted score first
        n = len(x)
        acc = np.zeros((n, 4)); fb = np.zeros((n, 4)); sw = np.zeros(n); best = np.zeros((n, M)); m = 0
        for x1, y1, x2, y2, s, _, k in x:
            k = int(k); ws = s * w[k]
            if m:
                f = fb[:m]
                ix = np.clip(np.minimum(f[:, 2], x2) - np.maximum(f[:, 0], x1), 0, None)
                iy = np.clip(np.minimum(f[:, 3], y2) - np.maximum(f[:, 1], y1), 0, None)
                inter = ix * iy
                iou = inter / ((f[:, 2] - f[:, 0]) * (f[:, 3] - f[:, 1]) + (x2 - x1) * (y2 - y1) - inter + 1e-9)
                iou[best[:m, k] > 0] = 0                               # at most one box per model in a fused box
                j = int(np.argmax(iou))
                if iou[j] > iou_thr:
                    acc[j] += ws * np.array([x1, y1, x2, y2]); sw[j] += ws; fb[j] = acc[j] / sw[j]
                    best[j, k] = s
                    continue
            acc[m] = ws * np.array([x1, y1, x2, y2]); sw[m] = ws; fb[m] = (x1, y1, x2, y2); best[m] = 0; best[m, k] = s; m += 1
        found = best[:m] > 0
        share = (found * w).sum(1)                                     # weight share of the models that found the box
        mean = (best[:m] * w).sum(1) / share                           # weighted mean score of those models
        out.append(np.c_[fb[:m], mean * share ** p, np.full(m, c)])
    if not out:
        return np.zeros((0, 6), np.float32)
    r = np.concatenate(out).astype(np.float32)
    return r[np.argsort(-r[:, 4])][:MAX_DET]

def fuse_task(args):
    iid, parts, w, iou_thr, p = args
    return iid, fuse(parts, w, iou_thr, p)

def fuse_all(outs, ids, w, iou_thr, p):
    # outs: list of {image_id: (N, 6)}, one per member
    empty = np.zeros((0, 6), np.float32)
    return dict(pmap(fuse_task, [(i, [o.get(i, empty) for o in outs], w, iou_thr, p) for i in ids], 16))

NAMES = list(VAL)
rows = []
def score(label, by, **extra):
    m = coco_eval(VAL_GT, to_dets(by), VAL_IDS)
    rows.append({"variant": label, **m, **extra})
    print(f"{label:46s} mAP {m['mAP']:.4f} | AP_s {m['AP_s']:.4f}", flush=True)
    return m["mAP"]

# each member alone, and the reproduction checks of notebooks 07 / 08 / 09
solo = {k: score(f"{k} alone", VAL[k]) for k in NAMES}
REF_04 = float(pd.read_csv(find("results_07_inference.csv")).set_index("strategy").loc["A2+A3_wbf", "mAP"])
REF_08 = float(pd.read_csv(find("results_08_summary.csv")).set_index("config").loc["final_08 A2+A3_wbf", "mAP"])
r9 = pd.read_csv(find("results_09_val.csv")).set_index("variant")
r9 = r9[r9.index.str.startswith("fusion")]
b9 = r9.mAP.idxmax(); REF_09 = float(r9.loc[b9, "mAP"])
w9 = float(r9.loc[b9, "w04"])
fus09 = score(f"09 fusion (04 + 08, {b9[7:]})", fuse_all([VAL["04"], VAL["08"]], VAL_IDS, [w9, 1 - w9], float(r9.loc[b9, "iou"]), float(r9.loc[b9, "p"])))
assert abs(solo["04"] - REF_04) < 0.005 and abs(solo["08"] - REF_08) < 0.005, "04 / 08 alone differ from notebooks 07 / 08"
assert abs(fus09 - REF_09) < 0.002, f"09's fusion gives {fus09:.4f} here, notebook 09 reported {REF_09:.4f}"

04 alone                                       mAP 0.2046 | AP_s 0.1645
08 alone                                       mAP 0.2066 | AP_s 0.1288
10 alone                                       mAP 0.2258 | AP_s 0.1450
09 fusion (04 + 08, w04=0.5 iou=0.6 p=2)       mAP 0.2250 | AP_s 0.1493


In [14]:
def try_fusion(w, iou_thr, p):
    label = "fusion " + " ".join(f"{k}={v:g}" for k, v in zip(NAMES, w)) + f" iou={iou_thr} p={p}"
    return score(label, fuse_all([VAL[k] for k in NAMES], VAL_IDS, w, iou_thr, p), w=list(w), iou=iou_thr, p=p)

tried = {}
def trial(w, iou_thr, p):
    key = (tuple(w), iou_thr, p)
    if key not in tried:
        tried[key] = try_fusion(list(w), iou_thr, p)
    return tried[key]

EQUAL = tuple([1.0] * len(NAMES))
# stage 1: penalty for boxes found by few models
P = max((1, 2, 3), key=lambda p: trial(EQUAL, 0.55, p))
# stage 2: one member up-weighted (x2) or left out, in turn
cands = [EQUAL] + [tuple(2.0 if j == i else 1.0 for j in range(len(NAMES))) for i in range(len(NAMES))] \
                + [tuple(0.0 if j == i else 1.0 for j in range(len(NAMES))) for i in range(len(NAMES))]
W = max(cands, key=lambda w: trial(w, 0.55, P))
# stage 3: matching IoU
IOU = max((0.55, 0.5, 0.6), key=lambda t: trial(W, t, P))
best_map = tried[(W, IOU, P)]
res_val = pd.DataFrame(rows).set_index("variant")
res_val.to_csv(WORK / "results_10_val.csv")
display(res_val.drop(columns=[c for c in ("w", "iou", "p") if c in res_val]).round(4))
print("best fusion:", dict(zip(NAMES, W)), f"iou={IOU} p={P} -> val mAP {best_map:.4f}")

fusion 04=1 08=1 10=1 iou=0.55 p=1             mAP 0.2417 | AP_s 0.1641
fusion 04=1 08=1 10=1 iou=0.55 p=2             mAP 0.2455 | AP_s 0.1707
fusion 04=1 08=1 10=1 iou=0.55 p=3             mAP 0.2460 | AP_s 0.1789
fusion 04=2 08=1 10=1 iou=0.55 p=3             mAP 0.2413 | AP_s 0.1788
fusion 04=1 08=2 10=1 iou=0.55 p=3             mAP 0.2423 | AP_s 0.1579
fusion 04=1 08=1 10=2 iou=0.55 p=3             mAP 0.2467 | AP_s 0.1812
fusion 04=0 08=1 10=1 iou=0.55 p=3             mAP 0.2407 | AP_s 0.1565
fusion 04=1 08=0 10=1 iou=0.55 p=3             mAP 0.2398 | AP_s 0.1866
fusion 04=1 08=1 10=0 iou=0.55 p=3             mAP 0.2251 | AP_s 0.1492
fusion 04=1 08=1 10=2 iou=0.5 p=3              mAP 0.2466 | AP_s 0.1812
fusion 04=1 08=1 10=2 iou=0.6 p=3              mAP 0.2469 | AP_s 0.1812


,mAP,AP50,AP_s,AP_m,AP_l
variant,,,,,
04 alone,0.2046,0.4058,0.1645,0.2059,0.3278
08 alone,0.2066,0.4159,0.1288,0.2127,0.3080
10 alone,0.2258,0.4353,0.1450,0.2165,0.3667
"09 fusion (04 + 08, w04=0.5 iou=0.6 p=2)",0.2250,0.4451,0.1493,0.2272,0.3389
fusion 04=1 08=1 10=1 iou=0.55 p=1,0.2417,0.4686,0.1641,0.2349,0.3813
fusion 04=1 08=1 10=1 iou=0.55 p=2,0.2455,0.4756,0.1707,0.2369,0.3852
fusion 04=1 08=1 10=1 iou=0.55 p=3,0.2460,0.4766,0.1789,0.2375,0.3852
fusion 04=2 08=1 10=1 iou=0.55 p=3,0.2413,0.4685,0.1788,0.2337,0.3789
fusion 04=1 08=2 10=1 iou=0.55 p=3,0.2423,0.4725,0.1579,0.2360,0.3769


best fusion: {'04': 1.0, '08': 1.0, '10': 2.0} iou=0.6 p=3 -> val mAP 0.2469


**Observations:**
- On the official val the larger model gains more than on fold 0: YOLO11m alone 0.226 vs 0.205 / 0.207 for the two YOLO11s pipelines, mainly from medium and large objects (AP_large 0.367 vs 0.31-0.33). AP_small is between them (0.145).
- With three members a stronger penalty for boxes found by few models works best (p = 3; 09 used p = 2). Every member helps: leaving one out costs 0.005 (04), 0.006 (08) or 0.021 (YOLO11m).
- Best fusion: 0.247, +0.022 over notebook 09's fusion. YOLO11m alone is about as good as 09's whole 2-model fusion.

### Test submission and verdict

In [15]:
fused = fuse_all([TEST[k] for k in NAMES], TEST_IDS, list(W), IOU, P)
write_submission(to_dets(fused), WORK / "submission_10_fusion.csv")

gain = best_map - REF_09
go = gain >= GO_MARGIN
print(f"\nval mAP: 09 fusion {REF_09:.4f} | YOLO11m alone {solo['10']:.4f} | best fusion {best_map:.4f} ({gain:+.4f} vs 09)")
print("VERDICT:", "GO -> submit submission_10_fusion.csv" if go else
      f"NO-GO -> gain below +{GO_MARGIN} on val: keep notebook 09's submission (0.2765)")

summary = pd.DataFrame(rows).assign(split="val")
if RUN_CV and len(folds_df):
    summary = pd.concat([folds_df.groupby("config")[METRICS].mean().reset_index().rename(columns={"config": "variant"}).assign(split="cv fold 0"),
                         summary], ignore_index=True)
summary.insert(0, "model", MODEL_NAME)
summary.to_csv(WORK / f"results_{NB}_summary.csv", index=False)

saved /kaggle/working/submission_10_fusion.csv: 3811 rows, 3811 with detections, 1861785 boxes
val mAP: 09 fusion 0.2250 | YOLO11m alone 0.2258 | best fusion 0.2469 (+0.0219 vs 09)
VERDICT: GO -> submit submission_10_fusion.csv


**Observations:**
- GO. Leaderboard 0.2869 vs 0.2765 for notebook 09 (+0.010 for +0.022 on val, about half transferred).
- Cost: ~3 h on one T4 (1 h fold-0 check, 1.1 h final training, the rest inference and fusion) for +0.010 on the leaderboard.